# Moving Average vs Naive Persistence Forecasting — Demo\n\nThis notebook investigates and quantifies the comparative performance between a 3-point moving average smoothing filter, an exponential weighted moving average (EWMA), and a naive last-value persistence forecasting model. The evaluation is conducted across a multi-seed experimental suite utilizing synthetic time-series data featuring complex dynamics (oscillatory autoinducer buffer behavior combined with Gaussian measurement noise and abrupt step changes).

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])\n\nif "google.colab" not in sys.modules:\n    _pip("numpy==2.0.2", "pandas==2.2.2", "matplotlib==3.10.0", "scipy==1.16.3")

In [ ]:
import numpy as np\nimport json\nimport os\nimport gc\nimport matplotlib.pyplot as plt\n\n# NumPy 2.0 compatibility shims if needed\nif not hasattr(np, "alltrue"): np.alltrue = np.all\nif not hasattr(np, "sometrue"): np.sometrue = np.any\nif not hasattr(np, "product"): np.product = np.prod\n\nprint("Imports loaded successfully.")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ca2cc5-resilient-quorum-sensing-multi-agent-rea/main/round-7/experiment-1/demo/mini_demo_data.json"\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception as e:\n        print(f"Network load failed ({e}), falling back to local file.")\n    if os.path.exists("mini_demo_data.json"):\n        with open("mini_demo_data.json") as f:\n            return json.load(f)\n    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local directory.")

In [ ]:
data = load_data()\nprint("Loaded metadata:", data.get("metadata", {}))\nprint("Available dataset keys:", [ds["dataset"] for ds in data.get("datasets", [])])

## Configuration Parameters\nDefine tunable parameters for synthetic data generation and model evaluation.

In [ ]:
# Config parameters (Tunable)\nNUM_STEPS = 200\nSEEDS = [42, 123, 456, 789, 1011]\nWARMUP = 10\nALPHA = 0.4\nMAX_EXAMPLES_PER_SEED = 20

## Time Series Forecasting Methods\nWe define functions for synthetic data generation (`generate_synthetic_series`), naive persistence forecasting (`naive_persistence`), 3-point moving average (`moving_average_3`), and exponential weighted moving average (`exponential_weighted_moving_average`).

In [ ]:
def generate_synthetic_series(num_steps=NUM_STEPS, seed=42):\n    np.random.seed(seed)\n    t = np.arange(num_steps)\n    base = np.sin(2 * np.pi * t / 50.0) * 10.0\n    steps = np.zeros(num_steps)\n    if num_steps >= 200:\n        steps[200:min(400, num_steps)] = 15.0\n    if num_steps >= 400:\n        steps[400:min(600, num_steps)] = -10.0\n    if num_steps >= 600:\n        steps[600:min(800, num_steps)] = 20.0\n    \n    noise = np.random.normal(0, 2.0, size=num_steps)\n    series = base + steps + noise\n    return series\n\ndef naive_persistence(series):\n    preds = np.roll(series, 1)\n    preds[0] = series[0]\n    return preds\n\ndef moving_average_3(series):\n    preds = np.zeros_like(series)\n    for t in range(len(series)):\n        if t == 0:\n            preds[t] = series[0]\n        elif t == 1:\n            preds[t] = series[0]\n        elif t == 2:\n            preds[t] = (series[0] + series[1]) / 2.0\n        else:\n            preds[t] = (series[t-1] + series[t-2] + series[t-3]) / 3.0\n    return preds\n\ndef exponential_weighted_moving_average(series, alpha=ALPHA):\n    preds = np.zeros_like(series)\n    curr = series[0]\n    for t in range(len(series)):\n        if t == 0:\n            preds[t] = series[0]\n        else:\n            curr = alpha * series[t-1] + (1 - alpha) * curr\n            preds[t] = curr\n    return preds

## Running Experiment Across Seeds\nWe execute the forecasting pipeline across all configured random seeds, generating synthetic time series and forecasting predictions for comparison.

In [ ]:
synthetic_examples = []\n\nfor seed in SEEDS:\n    series = generate_synthetic_series(num_steps=NUM_STEPS, seed=seed)\n    y_true = series[WARMUP:]\n    naive_preds = naive_persistence(series)[WARMUP:]\n    ma3_preds = moving_average_3(series)[WARMUP:]\n    ewma_preds = exponential_weighted_moving_average(series, alpha=ALPHA)[WARMUP:]\n    \n    for idx in range(min(MAX_EXAMPLES_PER_SEED, len(y_true))):\n        synthetic_examples.append({\n            "input": f"Synthetic time series forecast step {WARMUP + idx} for seed {seed}",\n            "output": str(float(y_true[idx])),\n            "predict_naive": str(float(naive_preds[idx])),\n            "predict_moving_average_3": str(float(ma3_preds[idx])),\n            "predict_ewma": str(float(ewma_preds[idx])),\n            "metadata_seed": str(seed),\n            "metadata_step": str(WARMUP + idx)\n        })\n        \n    del series, y_true, naive_preds, ma3_preds, ewma_preds\n    gc.collect()\n\nprint(f"Generated {len(synthetic_examples)} synthetic examples across {len(SEEDS)} seeds.")

## Results & Visualization\nWe evaluate and visualize the forecasting performance on a sample seed, comparing the true time series against Naive Persistence, 3-Point Moving Average, and EWMA predictions. We also compute Mean Squared Error (MSE) and Mean Absolute Error (MAE) across models.

In [ ]:
sample_seed = SEEDS[0]\nsample_series = generate_synthetic_series(num_steps=NUM_STEPS, seed=sample_seed)\ny_true = sample_series[WARMUP:]\np_naive = naive_persistence(sample_series)[WARMUP:]\np_ma3 = moving_average_3(sample_series)[WARMUP:]\np_ewma = exponential_weighted_moving_average(sample_series, alpha=ALPHA)[WARMUP:]\n\nmse_naive = np.mean((y_true - p_naive) ** 2)\nmae_naive = np.mean(np.abs(y_true - p_naive))\n\nmse_ma3 = np.mean((y_true - p_ma3) ** 2)\nmae_ma3 = np.mean(np.abs(y_true - p_ma3))\n\nmse_ewma = np.mean((y_true - p_ewma) ** 2)\nmae_ewma = np.mean(np.abs(y_true - p_ewma))\n\nprint("Model Performance Evaluation (Seed 42):")\nprint(f"  Naive Persistence: MSE = {mse_naive:.4f}, MAE = {mae_naive:.4f}")\nprint(f"  3-Point MA:        MSE = {mse_ma3:.4f}, MAE = {mae_ma3:.4f}")\nprint(f"  EWMA (alpha={ALPHA}): MSE = {mse_ewma:.4f}, MAE = {mae_ewma:.4f}")\n\nplt.figure(figsize=(12, 6))\nsteps_range = np.arange(WARMUP, WARMUP + len(y_true))\nplt.plot(steps_range[:80], y_true[:80], label="Ground Truth", color="black", linewidth=2)\nplt.plot(steps_range[:80], p_naive[:80], label="Naive Persistence", linestyle="--", color="red", alpha=0.7)\nplt.plot(steps_range[:80], p_ma3[:80], label="3-Point Moving Average", linestyle="-.", color="blue", alpha=0.8)\nplt.plot(steps_range[:80], p_ewma[:80], label=f"EWMA (alpha={ALPHA})", linestyle=":", color="green", alpha=0.8)\nplt.xlabel("Time Step")\nplt.ylabel("Value")\nplt.title(f"Forecasting Comparison on Synthetic Time Series (Seed {sample_seed})")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()